# Multiple Testing Data Generator

This notebook translates `data_generator.R`, `data_generator_PRDS.R`, and `generate_sim_data.R` from the follow-up paper "Multiple testing with anytime-valid Monte-Carlo p-values".
It contains the logic to generate simulated test statistics and apply different sequential multiple testing procedures.

In [1]:
import numpy as np
import pandas as pd
import pickle
import os
from sequential_BH import sequential_BH, p_adjust_fdr

os.makedirs("results", exist_ok=True)
np.random.seed(123)

## 1. Data Generator Function

In [2]:
def data_generator(nsim, nhyp, mu_As, mu_N, pis, alphas, param_bm, param_bc, param_amt, peeking_times_AMT, B_large, B_small, filename):
    """
    Simulates data and applies different procedures.
    To speed up the python execution, you may lower `nsim` or `B_large`.
    """
    if isinstance(nhyp, int): nhyp = [nhyp]
    if isinstance(alphas, float): alphas = [alphas]
    if isinstance(mu_As, (int, float)): mu_As = [mu_As]
    if isinstance(pis, float): pis = [pis]
        
    num_configs = len(pis) * len(alphas) * len(mu_As) * len(nhyp)
    
    # Initialize result arrays (averaged over nsim later)
    res_keys = ['power_BH', 'FDR_BH', 'power_BH_smallB', 'FDR_BH_smallB', 'power_bc_old', 'FDR_bc_old', 'nperm_mean_bc_old',
               'power_bm', 'FDR_bm', 'nperm_mean_bm', 'nperm_median_bm', 'nperm_max_bm', 'nperm_mean_rej_bm', 'nperm_mean_stop_bm',
               'power_bc', 'FDR_bc', 'nperm_mean_bc', 'nperm_median_bc', 'nperm_max_bc', 'nperm_mean_rej_bc', 'nperm_mean_stop_bc',
               'power_agg', 'FDR_agg', 'nperm_mean_agg', 'nperm_median_agg', 'nperm_max_agg', 'nperm_mean_rej_agg', 'nperm_mean_stop_agg',
               'power_amt', 'FDR_amt', 'nperm_mean_amt', 'nperm_median_amt', 'nperm_max_amt', 'nperm_mean_rej_amt', 'nperm_mean_stop_amt']
    
    results = {k: np.zeros((nsim, num_configs)) for k in res_keys}
    
    # To store detailed results for the last configuration
    result_bm_list = []
    result_bc_list = []
    
    count = 0
    for M in nhyp:
        for alpha in alphas:
            for mu_A in mu_As:
                for pi_A in pis:
                    print(f"Config: M={M}, alpha={alpha}, mu_A={mu_A}, pi_A={pi_A}")
                    Z = np.zeros((M, nsim))
                    hypo = np.zeros((M, nsim), dtype=int)
                    
                    for j in range(nsim):
                        hypo[:, j] = np.random.binomial(1, pi_A, M)
                        X = np.random.randn(M)
                        Z[:, j] = mu_N * (hypo[:, j] - 1) * (-1) + mu_A * hypo[:, j] + X
                        
                    for k in range(nsim):
                        simZ = np.random.randn(M, B_small)
                        simZ_largeB = np.random.randn(M, B_large)
                        
                        peeking_times = list(range(1, B_large + 1))
                        
                        # Apply algorithms
                        res_bm = sequential_BH(Z[:, k], simZ_largeB, M, B_large, peeking_times, alpha, param_bm, param_bc, param_amt, peeking_times_AMT, "bm")
                        res_bc = sequential_BH(Z[:, k], simZ_largeB, M, B_large, peeking_times, alpha, param_bm, param_bc, param_amt, peeking_times_AMT, "bc")
                        res_agg = sequential_BH(Z[:, k], simZ_largeB, M, B_large, peeking_times, alpha, param_bm, 1, param_amt, peeking_times_AMT, "bc")
                        res_amt = sequential_BH(Z[:, k], simZ_largeB, M, B_large, peeking_times, alpha - param_amt, param_bm, param_bc, param_amt, peeking_times_AMT, "AMT")
                        
                        if k == 0: # store one for plotting distributions
                            result_bm_list.append(res_bm)
                            result_bc_list.append(res_bc)
                            
                        # Helper for metrics
                        def calc_metrics(res, pfx):
                            rej = res['rejects']
                            stop = res['stop']
                            results[f'nperm_mean_{pfx}'][k, count] = np.mean(stop)
                            if len(rej) > 0:
                                results[f'nperm_mean_rej_{pfx}'][k, count] = np.mean(stop[rej])
                            else:
                                results[f'nperm_mean_rej_{pfx}'][k, count] = 0
                                
                            non_rej = list(set(range(M)) - set(rej))
                            if len(non_rej) > 0:
                                results[f'nperm_mean_stop_{pfx}'][k, count] = np.mean(stop[non_rej])
                            else:
                                results[f'nperm_mean_stop_{pfx}'][k, count] = 0
                                
                            results[f'nperm_median_{pfx}'][k, count] = np.median(stop)
                            results[f'nperm_max_{pfx}'][k, count] = np.max(stop)
                            
                            true_H1 = np.where(hypo[:, k] == 1)[0]
                            true_H0 = np.where(hypo[:, k] == 0)[0]
                            
                            correct_rej = len(set(true_H1).intersection(rej))
                            false_rej = len(set(true_H0).intersection(rej))
                            
                            results[f'power_{pfx}'][k, count] = correct_rej / max(1, len(true_H1))
                            results[f'FDR_{pfx}'][k, count] = false_rej / max(1, len(rej))
                        
                        calc_metrics(res_bm, 'bm')
                        calc_metrics(res_bc, 'bc')
                        calc_metrics(res_agg, 'agg')
                        calc_metrics(res_amt, 'amt')
                        
                        # Classical perm
                        ps_perm = (np.sum(simZ_largeB >= Z[:, k, None], axis=1) + 1) / (B_large + 1)
                        ps_perm_BH_rej = p_adjust_fdr(ps_perm) <= alpha
                        rej_BH = np.where(ps_perm_BH_rej)[0]
                        
                        true_H1 = np.where(hypo[:, k] == 1)[0]
                        true_H0 = np.where(hypo[:, k] == 0)[0]
                        
                        results['power_BH'][k, count] = len(set(true_H1).intersection(rej_BH)) / max(1, len(true_H1))
                        results['FDR_BH'][k, count] = len(set(true_H0).intersection(rej_BH)) / max(1, len(rej_BH))
                        
                        # Small B
                        ps_perm_smallB = (np.sum(simZ >= Z[:, k, None], axis=1) + 1) / (B_small + 1)
                        ps_perm_BH_smallB_rej = p_adjust_fdr(ps_perm_smallB) <= alpha
                        rej_BH_smallB = np.where(ps_perm_BH_smallB_rej)[0]
                        
                        results['power_BH_smallB'][k, count] = len(set(true_H1).intersection(rej_BH_smallB)) / max(1, len(true_H1))
                        results['FDR_BH_smallB'][k, count] = len(set(true_H0).intersection(rej_BH_smallB)) / max(1, len(rej_BH_smallB))
                        
                        # Classical BC
                        cum_greater = np.cumsum(simZ_largeB >= Z[:, k, None], axis=1)
                        # which >= param_bc
                        time_fut = np.zeros(M, dtype=int)
                        for u in range(M):
                            idx = np.where(cum_greater[u] >= param_bc)[0]
                            time_fut[u] = idx[0] + 1 if len(idx) > 0 else B_large + 1
                            
                        ps_bc_old = np.where(time_fut <= B_large, param_bc / time_fut, ps_perm)
                        time_fut = np.minimum(time_fut, B_large)
                        
                        ps_bc_old_BH_rej = p_adjust_fdr(ps_bc_old) <= alpha
                        rej_bc_old = np.where(ps_bc_old_BH_rej)[0]
                        results['power_bc_old'][k, count] = len(set(true_H1).intersection(rej_bc_old)) / max(1, len(true_H1))
                        results['FDR_bc_old'][k, count] = len(set(true_H0).intersection(rej_bc_old)) / max(1, len(rej_bc_old))
                        results['nperm_mean_bc_old'][k, count] = np.mean(time_fut)

                    count += 1
                    
    # Average over simulations
    final_res = {k: np.mean(v, axis=0) for k, v in results.items()}
    final_res['result_bm_list'] = result_bm_list
    final_res['result_bc_list'] = result_bc_list
    
    with open(filename, 'wb') as f:
        pickle.dump(final_res, f)
        
    print(f"Saved results to {filename}")

## 2. Data Generator PRDS Function

In [3]:
def data_generator_PRDS(nsim, nhyp, mu_As, mu_N, pis, rhos, alphas, param_bm, param_bc, param_amt, peeking_times_AMT, B_large, B_small, filename):
    """
    Data generator for PRDS (positive dependence) test statistics
    """
    if isinstance(nhyp, int): nhyp = [nhyp]
    if isinstance(alphas, float): alphas = [alphas]
    if isinstance(mu_As, (int, float)): mu_As = [mu_As]
    if isinstance(pis, float): pis = [pis]
    if isinstance(rhos, (int, float)): rhos = [rhos]
        
    num_configs = len(pis) * len(alphas) * len(mu_As) * len(nhyp) * len(rhos)
    
    res_keys = ['FDR_BH', 'FDR_BH_smallB', 'FDR_bm', 'FDR_bc', 'FDR_agg',
                'FDR_BH_sd', 'FDR_BH_smallB_sd', 'FDR_bm_sd', 'FDR_bc_sd', 'FDR_agg_sd']
    results = {k: np.zeros((nsim, num_configs)) for k in res_keys}
    
    count = 0
    for M in nhyp:
        for alpha in alphas:
            for mu_A in mu_As:
                for pi_A in pis:
                    for rho in rhos:
                        print(f"Config PRDS: rho={rho}")
                        hypo = np.random.binomial(1, pi_A, (M, nsim))
                        
                        for k in range(nsim):
                            # Generate correlated data
                            cov = np.full((M, M), rho)
                            np.fill_diagonal(cov, 1)
                            X = np.random.multivariate_normal(np.zeros(M), cov)
                            Z_k = mu_N * (hypo[:, k] - 1) * (-1) + mu_A * hypo[:, k] + X
                            
                            simZ = np.random.randn(M, B_small)
                            simZ_largeB = np.random.randn(M, B_large)
                            peeking_times = list(range(1, B_large + 1))
                            
                            res_bm = sequential_BH(Z_k, simZ_largeB, M, B_large, peeking_times, alpha, param_bm, param_bc, param_amt, peeking_times_AMT, "bm")
                            res_bc = sequential_BH(Z_k, simZ_largeB, M, B_large, peeking_times, alpha, param_bm, param_bc, param_amt, peeking_times_AMT, "bc")
                            res_agg = sequential_BH(Z_k, simZ_largeB, M, B_large, peeking_times, alpha, param_bm, 1, param_amt, peeking_times_AMT, "bc")
                            
                            true_H0 = np.where(hypo[:, k] == 0)[0]
                            
                            results['FDR_bm'][k, count] = len(set(true_H0).intersection(res_bm['rejects'])) / max(1, len(res_bm['rejects']))
                            results['FDR_bc'][k, count] = len(set(true_H0).intersection(res_bc['rejects'])) / max(1, len(res_bc['rejects']))
                            results['FDR_agg'][k, count] = len(set(true_H0).intersection(res_agg['rejects'])) / max(1, len(res_agg['rejects']))
                            
                            ps_perm = (np.sum(simZ_largeB >= Z_k[:, None], axis=1) + 1) / (B_large + 1)
                            rej_BH = np.where(p_adjust_fdr(ps_perm) <= alpha)[0]
                            results['FDR_BH'][k, count] = len(set(true_H0).intersection(rej_BH)) / max(1, len(rej_BH))
                            
                            ps_perm_smallB = (np.sum(simZ >= Z_k[:, None], axis=1) + 1) / (B_small + 1)
                            rej_BH_smallB = np.where(p_adjust_fdr(ps_perm_smallB) <= alpha)[0]
                            results['FDR_BH_smallB'][k, count] = len(set(true_H0).intersection(rej_BH_smallB)) / max(1, len(rej_BH_smallB))
                            
                        count += 1
                        
    final_res = {}
    for k in ['FDR_BH', 'FDR_BH_smallB', 'FDR_bm', 'FDR_bc', 'FDR_agg']:
        final_res[k] = np.mean(results[k], axis=0)
        final_res[k + '_sd'] = np.std(results[k], axis=0) / np.sqrt(nsim)
        
    with open(filename, 'wb') as f:
        pickle.dump(final_res, f)
        
    print(f"Saved results to {filename}")

## 3. Generate Simulated Data
(We run abbreviated versions here to save time. Change `nsim` to 10 or `B_large` to 10000 for full paper reproduction)

In [4]:
peeking_times_AMT = np.floor(np.cumsum(np.concatenate(([100], 100 * np.cumprod(np.full(24, 1.1)))))).tolist() + [10000]

print("This section is commented out to avoid extremely long execution times in the notebook.")
print("To run the full simulations, uncomment the lines below.")

# data_generator(nsim=10, nhyp=1000, mu_As=2.5, mu_N=0, pis=list(np.arange(0.1, 1.0, 0.1)), alphas=0.1,
#               param_bm=0.9, param_bc=10, param_amt=0.01, peeking_times_AMT=peeking_times_AMT,
#               B_large=10000, B_small=200, filename="results/Plot_FDR_pis.pkl")

# data_generator(nsim=10, nhyp=1000, mu_As=2.5, mu_N=0, pis=0.4, alphas=[0.01, 0.05, 0.1, 0.15, 0.2],
#               param_bm=0.9, param_bc=10, param_amt=0.01, peeking_times_AMT=peeking_times_AMT,
#               B_large=10000, B_small=200, filename="results/Plot_FDR_alphas.pkl")

# data_generator(nsim=10, nhyp=1000, mu_As=[1, 1.5, 2, 2.5, 3, 3.5, 4], mu_N=0, pis=0.4, alphas=0.1,
#               param_bm=0.9, param_bc=10, param_amt=0.01, peeking_times_AMT=peeking_times_AMT,
#               B_large=10000, B_small=200, filename="results/Plot_FDR_mus.pkl")

# data_generator(nsim=10, nhyp=[100, 500, 1000, 3000, 5000, 10000], mu_As=2.5, mu_N=0, pis=0.4, alphas=0.1,
#               param_bm=0.9, param_bc=10, param_amt=0.01, peeking_times_AMT=peeking_times_AMT,
#               B_large=10000, B_small=200, filename="results/Plot_FDR_Ms.pkl")

# data_generator_PRDS(nsim=1000, nhyp=1000, mu_As=2.5, mu_N=0, pis=0.4, rhos=[0, 0.1, 0.3, 0.5, 0.7, 0.9], alphas=0.1,
#               param_bm=0.9, param_bc=10, param_amt=0.01, peeking_times_AMT=peeking_times_AMT,
#               B_large=10000, B_small=200, filename="results/Plot_FDR_rhos.pkl")

# data_generator(nsim=1, nhyp=1000, mu_As=2.5, mu_N=0, pis=0.4, alphas=0.1,
#               param_bm=0.9, param_bc=10, param_amt=0.01, peeking_times_AMT=peeking_times_AMT,
#               B_large=10000, B_small=200, filename="results/Plot_FDR_standard.pkl")

This section is commented out to avoid extremely long execution times in the notebook.
To run the full simulations, uncomment the lines below.
